Load Data

In [4]:
import os
os.getcwd()
os.chdir("/Users/mig/Documents/LSE/Capstone Project/lse-bitcoin-analytics-capstone-template/")

In [5]:
%run eda/eda_starter_template.py


[Memory] Initial memory usage: 275.38 MB

Loading Bitcoin data from /Users/mig/Documents/LSE/Capstone Project/lse-bitcoin-analytics-capstone-template/data/Coin Metrics/coinmetrics_btc.csv...
[Memory] Before loading Bitcoin data: 275.38 MB
[Memory] After loading Bitcoin data: 279.11 MB (Δ 3.73 MB)
Error loading Bitcoin data: No such file or directory (os error 2): ...ne Project/lse-bitcoin-analytics-capstone-template/data/Coin Metrics/coinmetrics_btc.csv (set POLARS_VERBOSE=1 to see full path)

This error occurred with the following context stack:
	[1] 'csv scan'
	[2] 'with_columns'
	[3] 'sink'

Loading Polymarket data from /Users/mig/Documents/LSE/Capstone Project/lse-bitcoin-analytics-capstone-template/data/Polymarket...
[Memory] Before loading Polymarket data: 279.12 MB
[Memory] After loading Polymarket data: 279.12 MB (Δ 0.00 MB)

[Memory] Final memory usage: 279.12 MB (Total Δ: 3.75 MB)

EDA Layout Complete. Check the 'plots' directory for visualizations.


In [6]:
import pandas as pd

COINMETRICS_PATH = '/Users/mig/Documents/LSE/Capstone Project/lse-bitcoin-analytics-capstone-template/eda/data/Coin Metrics/coinmetrics_btc.csv'
POLYMARKET_DIR = '/Users/mig/Documents/LSE/Capstone Project/lse-bitcoin-analytics-capstone-template/eda/data/Polymarket'

# Load Bitcoin data
btc_df = load_bitcoin_data(COINMETRICS_PATH)

# Debug: Check directory and files
print("\n" + "="*80)
print("POLYMARKET DATA LOADING")
print("="*80)
print(f"\nDirectory: {POLYMARKET_DIR}")
print(f"Directory exists: {os.path.exists(POLYMARKET_DIR)}")

if os.path.exists(POLYMARKET_DIR):
    print("\nFiles in directory:")
    all_files = os.listdir(POLYMARKET_DIR)
    for file in sorted(all_files):
        filepath = os.path.join(POLYMARKET_DIR, file)
        size = os.path.getsize(filepath) / (1024*1024)  # Size in MB
        print(f"  {file:50s} ({size:.2f} MB)")
else:
    print("\n✗ ERROR: Directory does not exist!")

print("\n" + "-"*80)

# Load Polymarket data with DETAILED error messages
poly_data = {}

files_to_load = [
    ('markets', 'finance_politics_markets.parquet'),
    ('odds_history', 'finance_politics_odds_history.parquet'),
    ('summary', 'finance_politics_summary.parquet'),
    ('tokens', 'finance_politics_tokens.parquet'), 
    ('trades', 'finance_politics_trades.parquet'),
    ('event_stats', 'finance_politics_event_stats.parquet')
]

for name, filename in files_to_load:
    filepath = os.path.join(POLYMARKET_DIR, filename)
    
    print(f"\n{name}:")
    print(f"  Looking for: {filename}")
    print(f"  Full path: {filepath}")
    print(f"  File exists: {os.path.exists(filepath)}")
    
    if not os.path.exists(filepath):
        print(f"  ✗ SKIPPED: File not found")
        continue
    
    try:
        poly_data[name] = pd.read_parquet(filepath)
        print(f"  ✓ SUCCESS: Loaded {len(poly_data[name]):,} rows, {len(poly_data[name].columns)} columns")
        print(f"  Columns: {', '.join(poly_data[name].columns[:5].tolist())}{'...' if len(poly_data[name].columns) > 5 else ''}")
        
    except Exception as e:
        print(f"  ✗ ERROR: {type(e).__name__}")
        print(f"  Message: {str(e)}")

# Final summary
print("\n" + "="*80)
print("SUMMARY")
print("="*80)
print(f"Bitcoin data: ✓ {len(btc_df):,} rows")
print(f"Polymarket datasets loaded: {len(poly_data)}/{len(files_to_load)}")

if len(poly_data) > 0:
    for name, df in poly_data.items():
        print(f"  ✓ {name:15s}: {len(df):,} rows")
else:
    print("\n⚠ WARNING: No Polymarket files loaded!")
    print("\nPossible reasons:")
    print("  1. Files have different names than expected")
    print("  2. Files are in a different directory")
    print("  3. Files haven't been downloaded yet")
    print("  4. pyarrow library not installed (run: pip install pyarrow)")

print("="*80)

Loading Bitcoin data from /Users/mig/Documents/LSE/Capstone Project/lse-bitcoin-analytics-capstone-template/eda/data/Coin Metrics/coinmetrics_btc.csv...
[Memory] Before loading Bitcoin data: 279.55 MB
[Memory] After loading Bitcoin data: 295.75 MB (Δ 16.20 MB)
Successfully loaded 6221 rows.

POLYMARKET DATA LOADING

Directory: /Users/mig/Documents/LSE/Capstone Project/lse-bitcoin-analytics-capstone-template/eda/data/Polymarket
Directory exists: True

Files in directory:
  finance_politics_event_stats.parquet               (1.37 MB)
  finance_politics_markets.parquet                   (3.22 MB)
  finance_politics_odds_history.parquet              (16.96 MB)
  finance_politics_summary.parquet                   (4.31 MB)
  finance_politics_tokens.parquet                    (9.87 MB)
  finance_politics_trades.parquet                    (3156.20 MB)
  polymarket_btc_analytics_schema.md                 (0.00 MB)

-------------------------------------------------------------------------------

In [8]:
os.chdir("/Users/mig/Documents/LSE/Capstone Project/lse-bitcoin-analytics-capstone-template/eda/")
classification_df = pd.read_csv('question_classifications.csv')
# UP keywords = question is asking if something goes up / happens positively
UP_KEYWORDS = [
    "reach", "hit", "above", "exceed", "over", "surpass", "rally",
    "rise", "increase", "grow", "gain", "approve", "win", "pass",
    "cut", "reduce", "lower", "deal", "agree", "sign", "launch",
    "bullish", "high", "top", "break"
]

# DOWN keywords = question is asking if something goes down / happens negatively
DOWN_KEYWORDS = [
    "below", "under", "drop", "fall", "crash", "dip", "decline",
    "lose", "ban", "reject", "fail", "cancel", "hike", "raise",
    "bearish", "low", "bottom", "collapse", "sell", "dump"
]

def detect_question_direction(question):
    q = question.lower()

    up_score   = sum(1 for kw in UP_KEYWORDS   if kw in q)
    down_score = sum(1 for kw in DOWN_KEYWORDS if kw in q)

    if up_score > down_score:
        return +1
    elif down_score > up_score:
        return -1
    else:
        return 0 
    
classification_df["question_direction"] = classification_df["question"].apply(
    detect_question_direction
)

In [108]:
poly_data_odds = poly_data["odds_history"]
poly_data_trades = poly_data["trades"]
poly_data_markets = poly_data["markets"]
poly_data_trades["date"] = pd.to_datetime(poly_data_trades["timestamp"].astype("int64"), unit="ms").dt.normalize()
print(poly_data_trades["date"].head())
trades_with_questions = poly_data_trades.merge(
    poly_data_markets,
    on="market_id",
    how="inner"
)
daily_odds = (
    trades_with_questions
    .groupby(["market_id", "question", "date"])["price"]
    .mean()
    .reset_index()
    .sort_values(["market_id", "date"])
)

0   2025-11-19
1   2025-11-19
2   2025-11-19
3   2025-11-19
4   2025-11-19
Name: date, dtype: datetime64[ns]


fed rate

In [109]:
fed_rate_bullish = classification_df[
    (classification_df['topic'] == 'Federal Reserve interest rate cut') & 
    (classification_df['question_direction'] == 1)]
match_fed_rate_bullish = fed_rate_bullish.merge(daily_odds, on='question', how='inner')

In [110]:
match_fed_rate_bullish['signal'] = match_fed_rate_bullish.groupby('question')['price'].diff()
match_fed_rate_bullish

,question,topic,confidence,btc_direction,question_direction,market_id,date,price,signal
0,Fed emergency rate cut in 2025?,Federal Reserve interest rate cut,0.736,1,1,516711,2025-11-26,0.692400,NaN
1,Fed emergency rate cut in 2025?,Federal Reserve interest rate cut,0.736,1,1,516711,2025-11-27,0.562221,-0.130179
2,Fed emergency rate cut in 2025?,Federal Reserve interest rate cut,0.736,1,1,516711,2025-11-28,0.554800,-0.007421
3,Fed emergency rate cut in 2025?,Federal Reserve interest rate cut,0.736,1,1,516711,2025-11-29,0.682112,0.127312
4,Fed emergency rate cut in 2025?,Federal Reserve interest rate cut,0.736,1,1,516711,2025-11-30,0.218050,-0.464062
...,...,...,...,...,...,...,...,...,...
1758,"Fed Derivative: Jan ""25bps cut"" over 30% on De...",Federal Reserve interest rate cut,0.847,1,1,813504,2025-12-05,0.232594,-0.040457
1759,"Fed Derivative: Jan ""25bps cut"" over 30% on De...",Federal Reserve interest rate cut,0.847,1,1,813504,2025-12-07,0.270974,0.038380
1760,"Fed Derivative: Jan ""25bps cut"" over 30% on De...",Federal Reserve interest rate cut,0.847,1,1,813504,2025-12-08,0.260000,-0.010974
1761,"Fed Derivative: Jan ""25bps cut"" over 30% on De...",Federal Reserve interest rate cut,0.847,1,1,813504,2025-12-09,0.505000,0.245000


eth

In [124]:
eth_bullish = classification_df[
    (classification_df['topic'] == 'Ethereum price target') & 
    (classification_df['question_direction'] == 1)]
match_eth_bullish = eth_bullish.merge(daily_odds, on='question', how='inner')

In [128]:
match_eth_bullish

,question,topic,confidence,btc_direction,question_direction,market_id,date,price
0,Doge ETF approved by July 31?,Ethereum price target,0.225,1,1,513651,2025-04-07,0.343333
1,Doge ETF approved by July 31?,Ethereum price target,0.225,1,1,513651,2025-04-08,0.326000
2,Doge ETF approved by July 31?,Ethereum price target,0.225,1,1,513651,2025-04-09,0.310368
3,Doge ETF approved by July 31?,Ethereum price target,0.225,1,1,513651,2025-04-10,0.295000
4,Doge ETF approved by July 31?,Ethereum price target,0.225,1,1,513651,2025-04-11,0.265000
...,...,...,...,...,...,...,...,...
6961,"Will the price of Ethereum be above $3,300 on ...",Ethereum price target,0.704,1,1,896103,2025-12-09,0.500000
6962,"Will the price of Ethereum be above $2,900 on ...",Ethereum price target,0.690,1,1,902604,2025-12-16,0.999000
6963,"Will the price of Ethereum be above $3,000 on ...",Ethereum price target,0.694,1,1,902605,2025-12-16,0.001000
6964,"Will the price of Ethereum be above $3,300 on ...",Ethereum price target,0.702,1,1,902608,2025-12-16,0.999000


In [125]:
eth_bearish = classification_df[
    (classification_df['topic'] == 'Ethereum price target') & 
    (classification_df['question_direction'] == -1)]
match_eth_bearish = eth_bearish.merge(daily_odds, on='question', how='inner')

In [146]:
print(type(eth_bullish_date_mean.index))

<class 'pandas.core.indexes.datetimes.DatetimeIndex'>


In [148]:
eth_bullish_date = match_eth_bullish.sort_values(by='date')
eth_bullish_date_mean = eth_bullish_date.groupby('date')['price'].mean().clip(-1,1)
eth_bullish_date_mean.index = pd.to_datetime(eth_bullish_date_mean.index)
eth_bullish_date_mean = eth_bullish_date_mean.rename("eth_bullish_signal")
eth_bullish_date_mean_change = eth_bullish_date_mean.diff()
eth_bullish_date_mean_change

date
2025-04-07         NaN
2025-04-08   -0.004060
2025-04-09   -0.002165
2025-04-10   -0.001283
2025-04-11    0.005814
                ...   
2025-12-31   -0.051231
2026-01-01   -0.070366
2026-01-02   -0.072571
2026-01-03    0.021017
2026-01-05    0.199985
Name: eth_bullish_signal, Length: 257, dtype: float64

In [ ]:
fed_signal_date_mean.name = "fed_signal"
fed_signal_date_mean.to_csv('fed_signal.csv', header=True)

In [111]:
print(match_fed_rate_bullish['date'].min())
print(match_fed_rate_bullish['date'].max())

2025-04-07 00:00:00
2026-01-05 00:00:00


In [112]:
PRICE_COL = "PriceUSD_coinmetrics"

# Strategy parameters
MIN_W = 1e-6
DYNAMIC_STRENGTH = 2.0  # Multiplier for weight adjustments

# Feature column names (for compatibility)
FEATS = [
    "fed_signal",
]

In [113]:
def precompute_features(df: pd.DataFrame, fed_signal: pd.Series) -> pd.DataFrame:
    """Compute fed rate feature for weight calculation.

    Features (all lagged 1 day to prevent look-ahead bias):
    - fed_signal: signal from fed rate, clipped to [-1, 1]

    Args:
        df: DataFrame with price column

    Returns:
        DataFrame with price and computed features
    """
    # Filter to valid date range
    price = df[PRICE_COL].loc["2010-07-18":].copy()

    # align fed signal to bitcoin date index — fills 0 where no signal exists
    fed_aligned = fed_signal.reindex(price.index).fillna(0)

    # Build and lag features
    features = pd.DataFrame(
        {
            PRICE_COL: price,
            "fed_signal": fed_aligned.shift(1).fillna(0),
        },
        index=price.index,
    )

    return features

In [114]:
import numpy as np
def compute_dynamic_multiplier(fed_signal: np.ndarray) -> np.ndarray:
    """Compute weight multiplier from fed signal.

    Simple strategy: use weight of fed rate signal

    Returns:
        Multipliers centered around 1.0
    """
    # Scale and clip
    adjustment = fed_signal * DYNAMIC_STRENGTH
    adjustment = np.clip(adjustment, -3, 3)

    multiplier = np.exp(adjustment)
    return np.where(np.isfinite(multiplier), multiplier, 1.0)

In [115]:
fed_signal_date = match_fed_rate_bullish[['date','signal']]

In [123]:
fed_signal_date = fed_signal_date.sort_values(by='date')
fed_signal_date_mean = fed_signal_date.groupby('date')['signal'].mean().clip(-1,1)
fed_signal_date_mean.name = "fed_signal"
fed_signal_date_mean.to_csv('fed_signal.csv', header=True)


In [117]:
print(type(btc_df))
print(btc_df.head())

<class 'pandas.core.frame.DataFrame'>
            AdrActCnt  AdrBalCnt  AssetCompletionTime  AssetEODCompletionTime  \
time                                                                            
2009-01-03        0.0        0.0         1.614335e+09            1.614335e+09   
2009-01-04        0.0        0.0         1.614335e+09            1.614335e+09   
2009-01-05        0.0        0.0         1.614335e+09            1.614335e+09   
2009-01-06        0.0        0.0         1.614335e+09            1.614335e+09   
2009-01-07        0.0        0.0         1.614335e+09            1.614335e+09   

            BlkCnt  CapMVRVCur  CapMrktCurUSD  CapMrktEstUSD  FeeTotNtv  \
time                                                                      
2009-01-03     0.0         NaN            NaN            NaN        0.0   
2009-01-04     0.0         NaN            NaN            NaN        0.0   
2009-01-05     0.0         NaN            NaN            NaN        0.0   
2009-01-06     0.0 

In [118]:
_FEATURES_DF = precompute_features(btc_df, fed_signal_date_mean)

In [119]:
def run_full_analysis(
    btc_df: pd.DataFrame,
    features_df: pd.DataFrame,
    compute_weights_fn,
    output_dir: Path | str,
    strategy_label: str = "Dynamic DCA",
    start_date = None,
    end_date = None,
):
    """Run full backtest analysis pipeline and generate all artifacts.

    Args:
        btc_df: DataFrame with PriceUSD_coinmetrics
        features_df: DataFrame with precomputed features
        compute_weights_fn: Function or callable that accepts (df_window, current_date)
        output_dir: Directory where charts and metrics.json will be saved
        strategy_label: Label for the strategy in charts
    """
    output_dir = Path(output_dir)
    os.makedirs(output_dir, exist_ok=True)

    logging.info(f"Running SPD backtest for '{strategy_label}'...")
    df_spd, exp_decay_percentile = backtest_dynamic_dca(
        btc_df,
        compute_weights_fn,
        features_df=features_df,
        strategy_label=strategy_label,
        start_date = start_date,
        end_date = end_date,
    )

    logging.info("Running strategy validation...")
    check_strategy_submission_ready(btc_df, compute_weights_fn)

    # Calculate metrics
    win_rate = (
        df_spd["dynamic_percentile"] > df_spd["uniform_percentile"]
    ).mean() * 100
    score = 0.5 * win_rate + 0.5 * exp_decay_percentile

    excess_percentile = df_spd["dynamic_percentile"] - df_spd["uniform_percentile"]
    mean_excess = excess_percentile.mean()
    median_excess = excess_percentile.median()

    uniform_pct_safe = df_spd["uniform_percentile"].replace(0, 0.01)
    relative_improvements = excess_percentile / uniform_pct_safe * 100
    relative_improvement_pct_mean = relative_improvements.mean()
    relative_improvement_pct_median = relative_improvements.median()

    wins = (df_spd["dynamic_percentile"] > df_spd["uniform_percentile"]).sum()
    losses = len(df_spd) - wins

    metrics = {
        "score": score,
        "win_rate": win_rate,
        "exp_decay_percentile": exp_decay_percentile,
        "mean_excess": mean_excess,
        "median_excess": median_excess,
        "relative_improvement_pct_mean": relative_improvement_pct_mean,
        "relative_improvement_pct_median": relative_improvement_pct_median,
        "mean_ratio": (
            df_spd["dynamic_percentile"] / df_spd["uniform_percentile"]
        ).mean(),
        "median_ratio": (
            df_spd["dynamic_percentile"] / df_spd["uniform_percentile"]
        ).median(),
        "total_windows": len(df_spd),
        "wins": int(wins),
        "losses": int(losses),
    }

    logging.info(f"Final Model Score: {score:.2f}%")
    logging.info(
        f"  Excess percentile: mean={mean_excess:.2f}%, median={median_excess:.2f}%"
    )
    logging.info(
        f"  Relative improvement: mean={relative_improvement_pct_mean:.2f}%, "
        f"median={relative_improvement_pct_median:.2f}%"
    )
    logging.info(
        f"  Ratio (dynamic/uniform): mean={metrics['mean_ratio']:.2f}, "
        f"median={metrics['median_ratio']:.2f}"
    )

    logging.info("Generating visualizations...")
    create_performance_comparison_chart(df_spd, output_dir)
    create_excess_percentile_distribution(df_spd, output_dir)
    create_win_loss_comparison(df_spd, output_dir)
    create_cumulative_performance(df_spd, output_dir)
    create_performance_metrics_summary(df_spd, metrics, output_dir)
    export_metrics_json(df_spd, metrics, output_dir)

    logging.info(f"All outputs saved to '{output_dir}/' directory")


In [120]:
btc_df = btc_df.rename(columns={"PriceUSD": "PriceUSD_coinmetrics"})

In [121]:
run_full_analysis(
    btc_df=btc_df,
    features_df=_FEATURES_DF,
    compute_weights_fn=compute_weights_modal,
    output_dir="output/fed_signal",
    strategy_label="Fed Signal DCA",
    start_date="2025-01-01",
    end_date="2026-01-31",
)

2026-04-12 14:40:46 INFO     Running SPD backtest for 'Fed Signal DCA'...
2026-04-12 14:40:46 INFO     Backtesting date range: 2025-01-01 to 2026-01-31 (31 total windows)
2026-04-12 14:40:47 INFO     ✓ Validated weight sums for 31 windows (all sum to 1.0)
2026-04-12 14:40:47 INFO     Aggregated Metrics for Fed Signal DCA:
2026-04-12 14:40:47 INFO       SPD: min=1018.84, max=1022.39, mean=1020.08, median=1019.05
2026-04-12 14:40:47 INFO       Percentile: min=42.81%, max=43.50%, mean=43.05%, median=42.85%
2026-04-12 14:40:47 INFO       Exp-decay avg SPD percentile: 42.90%
2026-04-12 14:40:47 INFO     Running strategy validation...


Validating strategy submission readiness...


2026-04-12 14:40:54 INFO     Backtesting date range: 2018-01-01 to 2025-12-31 (2557 total windows)
2026-04-12 14:40:59 INFO     ✓ Validated weight sums for 2557 windows (all sum to 1.0)



⚠️ Windows where strategy underperformed Uniform DCA:


,dynamic_percentile,uniform_percentile,Delta
window,,,
2018-10-27 → 2019-10-27,41.979515,42.184367,-0.204851
2018-10-29 → 2019-10-29,41.805001,42.070124,-0.265122
2018-10-30 → 2019-10-30,42.007191,42.011404,-0.004213
2018-10-31 → 2019-10-31,41.881028,41.953332,-0.072304
2018-11-01 → 2019-11-01,41.548311,41.894946,-0.346635
...,...,...,...
2024-09-08 → 2025-09-08,28.345757,28.747581,-0.401824
2024-09-09 → 2025-09-09,30.451677,30.699404,-0.247726
2024-09-10 → 2025-09-10,30.543384,30.718194,-0.174810


2026-04-12 14:40:59 INFO     Final Model Score: 71.45%
2026-04-12 14:40:59 INFO       Excess percentile: mean=4.04%, median=3.83%
2026-04-12 14:40:59 INFO       Relative improvement: mean=10.36%, median=9.83%
2026-04-12 14:40:59 INFO       Ratio (dynamic/uniform): mean=1.10, median=1.10
2026-04-12 14:40:59 INFO     Generating visualizations...
2026-04-12 14:40:59 INFO     ✓ Saved: output/fed_signal/performance_comparison.svg
2026-04-12 14:40:59 INFO     ✓ Saved: output/fed_signal/excess_percentile_distribution.svg



Summary: 996/2557 underperformed (61.05% win rate)
✅ Strategy meets performance requirement (≥ 50% win rate vs. uniform DCA).

✅ Strategy is ready for submission.


2026-04-12 14:40:59 INFO     ✓ Saved: output/fed_signal/win_loss_comparison.svg
2026-04-12 14:40:59 INFO     ✓ Saved: output/fed_signal/cumulative_performance.svg
2026-04-12 14:41:00 INFO     ✓ Saved: output/fed_signal/metrics_summary.svg
2026-04-12 14:41:00 INFO     ✓ Saved: output/fed_signal/metrics.json
2026-04-12 14:41:00 INFO     All outputs saved to 'output/fed_signal/' directory
